# K-Nearest Neighbors (kNN) Algorithm
## Dataset: UCI Adult Census Income

**Sinh viên:** Ngô Lâm Tiên — MSSV: 524H0195

---

## Giới thiệu Dataset

**Nguồn:** https://archive.ics.uci.edu/dataset/2/adult

**Mục tiêu:** Dự đoán thu nhập của một người có **>50K** hay **≤50K** USD/năm dựa trên các đặc trưng nhân khẩu học.

**Lý do chọn dataset này:** Dataset Adult Census Income chứa **cả đặc trưng số (numerical) và đặc trưng phân loại (categorical)**, rất phù hợp để minh hoạ quy trình tiền xử lý dữ liệu đầy đủ trước khi áp dụng kNN.

---

## Các đặc trưng (Features)

| Cột | Loại | Ý nghĩa |
|-----|------|----------|
| age | Numerical | Tuổi |
| workclass | Categorical | Loại hình công việc |
| fnlwgt | Numerical | Trọng số dân số |
| education | Categorical | Trình độ học vấn |
| education-num | Numerical | Số năm học |
| marital-status | Categorical | Tình trạng hôn nhân |
| occupation | Categorical | Nghề nghiệp |
| relationship | Categorical | Quan hệ gia đình |
| race | Categorical | Chủng tộc |
| sex | Categorical | Giới tính |
| capital-gain | Numerical | Thu nhập từ vốn |
| capital-loss | Numerical | Mất mát từ vốn |
| hours-per-week | Numerical | Số giờ làm/tuần |
| native-country | Categorical | Quốc gia gốc |
| **income** | **Label** | **>50K hoặc ≤50K** |

---

## Thuật toán kNN

Thuật toán **k-Nearest Neighbors (kNN)** phân loại một điểm dữ liệu mới dựa trên nhãn của **k điểm gần nhất** trong tập huấn luyện.

**Khoảng cách Euclidean:**
$$d(x, x') = \sqrt{\sum_{i=1}^{n}(x_i - x'_i)^2}$$

**Quy tắc phân loại:** Nhãn được chọn theo **biểu quyết đa số** từ k láng giềng gần nhất.

---

## Quy trình xử lý dữ liệu

```
1. Load dữ liệu từ UCI
2. Khám phá dữ liệu (EDA)
3. Xử lý giá trị thiếu (Missing Values)
4. Mã hoá đặc trưng phân loại (Label Encoding / One-Hot Encoding)
5. Chuẩn hoá đặc trưng số (StandardScaler)
6. Chia tập train/test
7. Huấn luyện kNN
8. Đánh giá mô hình
9. Tìm k tối ưu
```

---
## Bước 1: Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

print('✅ Import thư viện thành công!')
print(f'pandas  version: {pd.__version__}')
print(f'numpy   version: {np.__version__}')

---
## Bước 2: Load dữ liệu từ UCI Adult Census Income

In [ ]:
# ---- Load UCI Adult Census Income dataset ----
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

col_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

df = pd.read_csv(url, names=col_names, sep=',', skipinitialspace=True)

print('=== UCI Adult Census Income Dataset ===')
print(f'Số mẫu    : {df.shape[0]:,}')
print(f'Số đặc trưng: {df.shape[1] - 1}')
print()
print('5 dòng đầu tiên:')
df.head()

---
## Bước 3: Khám phá dữ liệu (Exploratory Data Analysis)

In [ ]:
# Thông tin tổng quan
print('=== THÔNG TIN DATASET ===')
df.info()
print()
print('=== THỐNG KÊ MÔ TẢ (CỘT SỐ) ===')
df.describe()

In [ ]:
# Phân bố nhãn (Label Distribution)
print('=== PHÂN PHỐI NHÃN (income) ===')
label_counts = df['income'].value_counts()
print(label_counts)
print(f'\nTỷ lệ: {label_counts[0]/len(df)*100:.1f}% vs {label_counts[1]/len(df)*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Biểu đồ phân bố nhãn
label_counts.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'],
                  edgecolor='white', rot=0)
axes[0].set_title('Phân phối Nhãn (income)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Income')
axes[0].set_ylabel('Số lượng mẫu')
for bar, val in zip(axes[0].patches, label_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                  f'{val:,}', ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Biểu đồ phân bố tuổi theo nhãn
df[df['income'] == '<=50K']['age'].hist(ax=axes[1], alpha=0.6, bins=30,
                                          color='#3498db', label='<=50K')
df[df['income'] == '>50K']['age'].hist(ax=axes[1], alpha=0.6, bins=30,
                                         color='#e74c3c', label='>50K')
axes[1].set_title('Phân phối Tuổi theo Thu nhập', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tuổi')
axes[1].set_ylabel('Số lượng')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Xác định các cột số và cột phân loại
numerical_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain',
                   'capital-loss', 'hours-per-week']
categorical_cols = ['workclass', 'education', 'marital-status', 'occupation',
                    'relationship', 'race', 'sex', 'native-country']

print('=== PHÂN LOẠI ĐẶC TRƯNG ===')
print(f'Đặc trưng SỐ       ({len(numerical_cols)} cột): {numerical_cols}')
print(f'Đặc trưng PHÂN LOẠI ({len(categorical_cols)} cột): {categorical_cols}')
print()

# Hiển thị giá trị unique của các cột phân loại
print('=== GIÁ TRỊ ĐỘC ĐÁO CỦA ĐẶC TRƯNG PHÂN LOẠI ===')
for col in categorical_cols:
    vals = df[col].unique()
    print(f'{col:20s}: {list(vals)}')

---
## Bước 4: Xử lý giá trị thiếu (Missing Values)

Dataset Adult dùng ký hiệu `?` để đánh dấu giá trị thiếu.

In [ ]:
# Thay '?' bằng NaN
df_clean = df.replace('?', np.nan)

print('=== KIỂM TRA GIÁ TRỊ THIẾU ===')
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean) * 100).round(2)
missing_df = pd.DataFrame({'Số lượng thiếu': missing, 'Tỉ lệ (%)': missing_pct})
missing_df = missing_df[missing_df['Số lượng thiếu'] > 0]

if len(missing_df) > 0:
    print(missing_df)
    print()
    print('→ Chiến lược: Điền bằng mode (giá trị xuất hiện nhiều nhất) cho cột phân loại')
    
    # Điền giá trị thiếu bằng mode
    for col in categorical_cols:
        if df_clean[col].isnull().any():
            mode_val = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(mode_val)
            print(f'  Đã điền cột "{col}" bằng mode = "{mode_val}"')
else:
    print('Không có giá trị thiếu!')

print()
print(f'Số mẫu sau xử lý: {len(df_clean):,}')
print(f'Còn thiếu: {df_clean.isnull().sum().sum()} giá trị')

---
## Bước 5: Tiền xử lý dữ liệu (Data Preprocessing)

### 5.1 Mã hoá nhãn (Label Encoding)
Chuyển nhãn `income` thành 0 và 1.

In [ ]:
# ---- Mã hoá nhãn (Label) ----
le_target = LabelEncoder()
y = le_target.fit_transform(df_clean['income'])

print('=== MÃ HOÁ NHÃN (income) ===')
print(f'Các lớp: {le_target.classes_}')
print(f'Mã hoá : {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')
print(f'Phân bố: {np.bincount(y)} (0={le_target.classes_[0]}, 1={le_target.classes_[1]})')

# Tách features
X_raw = df_clean.drop(columns=['income'])
print(f'\nShape của X: {X_raw.shape}')

### 5.2 Mã hoá đặc trưng phân loại & Chuẩn hoá đặc trưng số

- **Categorical → One-Hot Encoding**: Tránh thứ tự giả tạo
- **Numerical → StandardScaler**: Chuẩn hoá về mean=0, std=1 (quan trọng cho kNN vì kNN dùng khoảng cách!)

In [ ]:
# ---- ColumnTransformer: xử lý cột số và cột phân loại riêng biệt ----

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

print('=== QUY TRÌNH TIỀN XỬ LÝ ===')
print(f'Đặc trưng SỐ   ({len(numerical_cols)} cột) → StandardScaler')
print(f'Đặc trưng PHÂN LOẠI ({len(categorical_cols)} cột) → OneHotEncoder')
print()
print('Lý do chuẩn hoá khi dùng kNN:')
print('  → kNN tính khoảng cách Euclidean giữa các điểm')
print('  → Nếu không chuẩn hoá, cột có giá trị lớn (vd: fnlwgt ~100K) sẽ')
print('    "áp đảo" cột có giá trị nhỏ (vd: age ~30), gây kết quả sai lệch')

---
## Bước 6: Chia tập Train / Test

In [ ]:
# ---- Chia dữ liệu 70% train, 30% test ----
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.3,
    random_state=42,
    stratify=y          # Giữ tỉ lệ nhãn trong cả 2 tập
)

print('=== CHIA TẬP DỮ LIỆU ===')
print(f'Tổng số mẫu  : {len(X_raw):,}')
print(f'Tập Train (70%): {len(X_train):,} mẫu')
print(f'Tập Test  (30%): {len(X_test):,} mẫu')
print()
print('Phân bố nhãn trong tập Train:')
print(f'  <=50K : {np.sum(y_train==0):,} ({np.sum(y_train==0)/len(y_train)*100:.1f}%)')
print(f'  >50K  : {np.sum(y_train==1):,} ({np.sum(y_train==1)/len(y_train)*100:.1f}%)')
print()
print('Phân bố nhãn trong tập Test:')
print(f'  <=50K : {np.sum(y_test==0):,} ({np.sum(y_test==0)/len(y_test)*100:.1f}%)')
print(f'  >50K  : {np.sum(y_test==1):,} ({np.sum(y_test==1)/len(y_test)*100:.1f}%)')

In [ ]:
# ---- Áp dụng tiền xử lý (fit trên train, transform cả train và test) ----
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

# Lấy tên feature sau One-Hot Encoding
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_feature_names = numerical_cols + list(cat_feature_names)

print('=== KẾT QUẢ SAU TIỀN XỬ LÝ ===')
print(f'Số đặc trưng ban đầu    : {X_raw.shape[1]}')
print(f'Số đặc trưng sau xử lý : {X_train_processed.shape[1]}')
print(f'  + Đặc trưng số (StandardScaler): {len(numerical_cols)}')
print(f'  + Đặc trưng One-Hot             : {len(cat_feature_names)}')
print()
print(f'Shape X_train sau xử lý: {X_train_processed.shape}')
print(f'Shape X_test  sau xử lý: {X_test_processed.shape}')

---
## Bước 7: Huấn luyện mô hình kNN (k=5)

In [ ]:
# ---- Huấn luyện KNN với k=5 ----
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric='euclidean',   # Khoảng cách Euclidean
    weights='uniform'     # Tất cả k láng giềng có trọng số bằng nhau
)

knn.fit(X_train_processed, y_train)
y_pred = knn.predict(X_test_processed)

print('=== KNN MODEL (k=5) ===')
print(f'Thuật toán : K-Nearest Neighbors')
print(f'k          : {knn.n_neighbors}')
print(f'Metric     : {knn.metric}')
print(f'Weights    : {knn.weights}')
print()
print(f'Dự đoán (10 mẫu đầu): {y_pred[:10]}')
print(f'Thực tế (10 mẫu đầu): {y_test[:10]}')

---
## Bước 8: Đánh giá mô hình

In [ ]:
# ============================================================
# ĐÁNH GIÁ MÔ HÌNH kNN
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
cm = confusion_matrix(y_test, y_pred)

# Tính TP, TN, FP, FN cho lớp >50K (lớp 1)
TN, FP, FN, TP = cm.ravel()

precision_1 = TP / (TP + FP) if (TP + FP) > 0 else 0
recall_1    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_1        = 2 * precision_1 * recall_1 / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0

print('======================================================')
print('     CLASSIFICATION METRICS — KNN (k=5)')
print('     Dataset: UCI Adult Census Income')
print('======================================================')
print(f'  Confusion Matrix:')
print(f'    TN = {TN:,}  (dự đoán đúng <=50K)')
print(f'    TP = {TP:,}  (dự đoán đúng >50K)')
print(f'    FP = {FP:,}  (dự đoán sai: <=50K nhưng đoán >50K)')
print(f'    FN = {FN:,}  (dự đoán sai: >50K nhưng đoán <=50K)')
print()
print(f'  Accuracy   = {accuracy:.4f} = {accuracy*100:.2f}%')
print(f'  Error Rate = {error_rate:.4f} = {error_rate*100:.2f}%')
print()
print(f'  Với lớp >50K:')
print(f'    Precision = {precision_1:.4f}')
print(f'    Recall    = {recall_1:.4f}')
print(f'    F1-score  = {f1_1:.4f}')
print()
print('======================================================')
print('CLASSIFICATION REPORT:')
print('======================================================')
class_names = ['<=50K', '>50K']
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# ---- Visualize kết quả đánh giá ----
from sklearn.metrics import precision_recall_fscore_support

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Plot 1: Confusion Matrix Heatmap ---
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix\n(kNN k=5, Accuracy = {accuracy*100:.1f}%)',
                   fontsize=12, fontweight='bold')
axes[0].set_xlabel('Dự đoán (Predicted Label)')
axes[0].set_ylabel('Thực tế (True Label)')

# --- Plot 2: Precision / Recall / F1 per class ---
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])
x = np.arange(len(class_names))
width = 0.25
axes[1].bar(x - width, prec, width, label='Precision', color='#3498db', alpha=0.85, edgecolor='white')
axes[1].bar(x,         rec,  width, label='Recall',    color='#e67e22', alpha=0.85, edgecolor='white')
axes[1].bar(x + width, f1,   width, label='F1-score',  color='#2ecc71', alpha=0.85, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(class_names)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel('Score')
axes[1].set_title('Precision / Recall / F1-score\ntheo từng lớp', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3, axis='y')
for i, (p, r, f) in enumerate(zip(prec, rec, f1)):
    axes[1].text(i - width, p + 0.02, f'{p:.2f}', ha='center', fontsize=10, fontweight='bold')
    axes[1].text(i,         r + 0.02, f'{r:.2f}', ha='center', fontsize=10, fontweight='bold')
    axes[1].text(i + width, f + 0.02, f'{f:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Model Evaluation — KNN (UCI Adult Census Income)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Bước 9: Tìm k tối ưu (Hyperparameter Tuning)

Thử nghiệm các giá trị k khác nhau để tìm ra k cho độ chính xác cao nhất.

In [ ]:
# ---- Tìm k tối ưu bằng cách thử nhiều giá trị k ----
k_values = list(range(1, 31, 2))   # k = 1, 3, 5, ..., 29 (số lẻ)
train_acc = []
test_acc  = []

print('Đang tính toán accuracy cho các giá trị k...')
for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_k.fit(X_train_processed, y_train)
    train_acc.append(accuracy_score(y_train, knn_k.predict(X_train_processed)))
    test_acc.append(accuracy_score(y_test,  knn_k.predict(X_test_processed)))
    if k % 5 == 1:
        print(f'  k={k:2d}: Train={train_acc[-1]:.4f}, Test={test_acc[-1]:.4f}')

best_k = k_values[np.argmax(test_acc)]
best_acc = max(test_acc)
print()
print(f'✅ k tối ưu = {best_k} với Test Accuracy = {best_acc:.4f} ({best_acc*100:.2f}%)')

In [ ]:
# ---- Visualize: Train vs Test Accuracy theo k ----
plt.figure(figsize=(10, 5))
plt.plot(k_values, train_acc, 'o-', color='#3498db', linewidth=2,
         markersize=6, label='Train Accuracy')
plt.plot(k_values, test_acc,  's-', color='#e74c3c', linewidth=2,
         markersize=6, label='Test Accuracy')
plt.axvline(x=best_k, color='#2ecc71', linestyle='--', linewidth=2,
            label=f'k tối ưu = {best_k}')
plt.axhline(y=best_acc, color='#2ecc71', linestyle=':', linewidth=1.5, alpha=0.7)
plt.scatter([best_k], [best_acc], color='#2ecc71', s=150, zorder=5)
plt.annotate(f'Best k={best_k}\nAcc={best_acc:.4f}',
             xy=(best_k, best_acc),
             xytext=(best_k + 2, best_acc - 0.015),
             fontsize=10, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='black'))
plt.xlabel('Giá trị k', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Train vs Test Accuracy theo k (kNN)\nDataset: UCI Adult Census Income',
          fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('NHẬN XÉT:')
print('  → Khi k nhỏ (k=1): Train Accuracy gần 100% nhưng Test thấp → Overfitting')
print('  → Khi k lớn hơn: Train giảm, Test có thể tăng → Mô hình tổng quát hơn')
print(f'  → k={best_k} cho Test Accuracy tốt nhất: {best_acc*100:.2f}%')

---
## Bước 10: Huấn luyện lại với k tối ưu & Kết quả cuối cùng

In [ ]:
# ---- Huấn luyện lại với k tối ưu ----
knn_best = KNeighborsClassifier(
    n_neighbors=best_k,
    metric='euclidean',
    weights='uniform'
)
knn_best.fit(X_train_processed, y_train)
y_pred_best = knn_best.predict(X_test_processed)

acc_best   = accuracy_score(y_test, y_pred_best)
cm_best    = confusion_matrix(y_test, y_pred_best)

print('======================================================')
print(f'  KNN MÔ HÌNH TỐI ƯU — k = {best_k}')
print('======================================================')
print(classification_report(y_test, y_pred_best, target_names=class_names))

# So sánh k=5 và k tối ưu
print('======================================================')
print('  SO SÁNH k=5 vs k tối ưu')
print('======================================================')
print(f'  k=5 (mặc định)  : Accuracy = {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  k={best_k} (tối ưu): Accuracy = {acc_best:.4f} ({acc_best*100:.2f}%)')
print(f'  Cải thiện        : +{(acc_best - accuracy)*100:.2f}%')

In [ ]:
# ---- Visualize kết quả cuối cùng ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Plot 1: Confusion Matrix (k tối ưu) ---
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=class_names)
disp2.plot(ax=axes[0], cmap='Greens', colorbar=False)
axes[0].set_title(f'Confusion Matrix — kNN (k={best_k})\nAccuracy = {acc_best*100:.1f}%',
                   fontsize=12, fontweight='bold')
axes[0].set_xlabel('Dự đoán (Predicted Label)')
axes[0].set_ylabel('Thực tế (True Label)')

# --- Plot 2: So sánh Accuracy ---
k_comp  = [5, best_k]
acc_comp = [accuracy, acc_best]
bars = axes[1].bar([f'k=5\n(mặc định)', f'k={best_k}\n(tối ưu)'],
                    acc_comp, color=['#3498db', '#2ecc71'],
                    edgecolor='white', width=0.5)
for bar, val in zip(bars, acc_comp):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                  bar.get_height() + 0.002,
                  f'{val*100:.2f}%', ha='center',
                  fontsize=13, fontweight='bold')
axes[1].set_ylim(0.75, 0.90)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('So sánh Accuracy\nk=5 vs k tối ưu', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Kết quả KNN Tối ưu — UCI Adult Census Income (k={best_k})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Tổng kết

### Quy trình đã thực hiện:

| Bước | Nội dung | Chi tiết |
|------|----------|----------|
| 1 | Load dữ liệu | UCI Adult Census Income (32,561 mẫu, 14 đặc trưng) |
| 2 | EDA | Phân tích phân bố nhãn, đặc trưng |
| 3 | Xử lý giá trị thiếu | Thay `?` → Mode |
| 4 | Mã hoá nhãn | LabelEncoder: `<=50K`→0, `>50K`→1 |
| 5 | Tiền xử lý features | **StandardScaler** (6 cột số) + **OneHotEncoder** (8 cột phân loại) |
| 6 | Chia train/test | 70% / 30% với stratify |
| 7 | Huấn luyện kNN | k=5, metric=Euclidean |
| 8 | Đánh giá | Accuracy, Confusion Matrix, Precision/Recall/F1 |
| 9 | Tối ưu k | Thử k từ 1→29, tìm k tốt nhất |

### Kết quả:

- **kNN k=5**: Accuracy ≈ 82-84%
- **kNN k tối ưu**: Cải thiện thêm ~1-2%

### Nhận xét quan trọng:

> **Tại sao phải chuẩn hoá dữ liệu (StandardScaler) trước khi dùng kNN?**
>
> - kNN sử dụng **khoảng cách Euclidean** để tìm láng giềng gần nhất
> - Nếu không chuẩn hoá: cột `fnlwgt` (giá trị ~100,000) sẽ **áp đảo** cột `age` (giá trị ~40)
> - Sau StandardScaler: tất cả đặc trưng có mean=0, std=1 → **đóng góp bình đẳng** vào khoảng cách

> **Tại sao dùng One-Hot Encoding cho đặc trưng phân loại?**
>
> - Đặc trưng như `workclass`, `occupation` không có thứ tự tự nhiên
> - Label Encoding sẽ tạo ra thứ tự giả tạo (vd: Private=0 < Self-emp=1 < Gov=2)
> - One-Hot Encoding tạo biến nhị phân cho mỗi giá trị → Không có thứ tự giả tạo